# Experiment 2 — MaleficNet SNR / spread-spectrum BER

Rebuilds **Table 5** and **Fig 3** (`fig:exp2_diff_new`): the signal quality of a
spread-spectrum payload hidden in model parameters, measured **clean** vs.
**after NeuPerm**. NeuPerm reorders the neurons the payload is keyed to, so the
correlator can no longer recover it — SNR (and extraction BER) collapse.

Everything here is rebuilt from the single canonical CSV `../results/exp2_snr.csv`.
Runs top-to-bottom with only `pandas` / `numpy` / `matplotlib` — no GPU, no
external files.

> **Reproduction note.** This notebook does **not** re-run the raw
> MaleficNet-fork SNR measurement. A from-scratch raw re-run of the fork-based
> SNR (`MODE = "maleficnet_fork"` in `experiments/exp2_maleficnet_snr.py`)
> requires the external MIT MaleficNet fork (`NEUPERM_MALEFICNET_DIR`) plus a
> payload directory — see `../payloads/README.md` and the experiment script's
> header. The table and figure below rebuild entirely from the shipped
> `exp2_snr.csv`, which unions every per-model / per-payload SNR run plus the
> self-contained spread-spectrum BER ablation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)

RESULTS_DIR = Path("../results")

MODEL_RENAME = {
    "densenet121": "DenseNet121", "resnet50": "ResNet50", "resnet101": "ResNet101",
    "vgg11": "VGG11", "vgg16": "VGG16", "mobilenet_v2": "MobileNetV2",
    "mobilenet_v3_small": "MobileNetV3-S",
    "efficientnet_b0": "EfficientNet-B0", "efficientnet_b4": "EfficientNet-B4",
    "llama-3.2-1b": "Llama-3.2-1B", "qwen2.5-1.5b": "Qwen2.5-1.5B",
}

snr = pd.read_csv(RESULTS_DIR / "exp2_snr.csv")
print("modes:", sorted(snr["mode"].unique()))
print("metrics:", sorted(snr["metric"].unique()))
snr.head()

## Table 5 — MaleficNet SNR, clean vs. after NeuPerm

`mode == "maleficnet_fork"`, `metric == "snr"`. One row per `(model, payload)`;
`condition` is `clean` or `after_neuperm`. `delta = after_neuperm - clean`;
negative means NeuPerm degraded the hidden signal. Some models have only `clean`
measurements in the shipped CSV (no `after_neuperm`), shown as `NaN` delta.

In [ ]:
mf = snr[
    (snr["mode"] == "maleficnet_fork")
    & (snr["metric"] == "snr")
    & (snr["condition"].isin(["clean", "after_neuperm"]))
].copy()
mf["Model"] = mf["model"].map(MODEL_RENAME)

table5 = mf.pivot_table(
    index=["Model", "payload"], columns="condition", values="value", aggfunc="mean"
).reindex(columns=["clean", "after_neuperm"])
table5["delta"] = table5["after_neuperm"] - table5["clean"]
table5

### Spread-spectrum extraction BER (self-contained ablation)

`mode == "spread_spectrum_llm"`: our independent spread-spectrum embed/extract on
Llama / Qwen (no external fork, no real malware). Bit-error-rate near 0 means
perfect recovery; NeuPerm drives it toward 0.5 (random).

In [ ]:
ber = snr[snr["metric"] == "ber"].copy()
ber["cond"] = ber["condition"].str.replace("ber_", "", regex=False)
ber["Model"] = ber["model"].map(MODEL_RENAME)
(
    ber.groupby(["Model", "cond"])["value"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

## Fig 3 — SNR change after NeuPerm (per model / payload)

For every `(model, payload)` pair that has both a `clean` and an `after_neuperm`
SNR, plot the difference `after_neuperm - clean`. Bars point left (negative) when
NeuPerm degraded the payload's recoverable signal.

In [ ]:
diff = table5.dropna(subset=["clean", "after_neuperm"]).reset_index()
diff = diff.sort_values("delta").reset_index(drop=True)
labels = [f"{m} / {p}" for m, p in zip(diff["Model"], diff["payload"])]

fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * len(diff))))
colors = np.where(diff["delta"] < 0, "#2b8cbe", "#e34a33")
ax.barh(range(len(diff)), diff["delta"], color=colors)
ax.set_yticks(range(len(diff)))
ax.set_yticklabels(labels, fontsize=7)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("SNR change after NeuPerm  (after_neuperm - clean)")
ax.set_title("Fig 3: Hidden-payload SNR collapses after NeuPerm")
fig.tight_layout()
plt.show()

print(f"{len(diff)} model/payload pairs; "
      f"{(diff['delta'] < 0).sum()} show reduced SNR after NeuPerm")